# Preprocessing

<div style="text-align: justify">
The following notebook is dedicated to preprocessing for the  <b> Tau Supersymmetry Anomaly Detection </b> analysis. Preprocessing involves handling .ROOT files for use in the anomaly detection pipeline. A general data processing workflow is structured as follows:
</div>

<img src="../assets/data_processing.png" alt="Workflow" style="width: 60%; display: block; margin: 0 auto;"/>

## Pipeline Summary

| Step | Module | Description |
|------|--------|-------------|
| Config | `hydra.compose` | Load analysis configuration |
| Resolve | `analysis.resolve_samples` | Build sample lists from config |
| Process | `processor.process_samples` | ROOT → cuts → weights per campaign |
| Merge BG | `merger.merge_backgrounds` | Group backgrounds by strategy |
| Label | `features.assign_event_origin` | Tag each event with its sample name |
| Merge SG | `merger.merge_signals` | Group signal samples by strategy |
| Save | `io.save_samples` | Write background, signal, and data parquet files |

The same pipeline is available as a CLI via `uv run python run.py stage=preprocess`.

## Initialization

### Libraries

Configuration:
* [Hydra](https://hydra.cc/)
* [OmegaConf](https://omegaconf.readthedocs.io/)
* [pyrootutils](https://github.com/ashleve/pyrootutils)

Data I/O:
* [UpROOT](https://uproot.readthedocs.io/en/latest/)
* [Awkward Array](https://awkward-array.readthedocs.io/en/latest/)

Serialization:
* [Apache Parquet](https://parquet.apache.org/)

### Notebook

Activating autoreload of imported modules.

In [1]:
%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format = "retina"

Initializing the project root.

In [2]:
import pyrootutils

path = pyrootutils.setup_root(
    search_from=__file__ if "__file__" in locals() else ".",
    indicator=".gitignore",
    pythonpath=True,
)

Suppressing unessential warnings and applying ATLAS style.

In [3]:
from src.utils import suppress_warnings
from src.visualization.plots import apply_atlas_style

suppress_warnings()
apply_atlas_style()

## Overview

List of available analyses.

<br>

<hr>

Run 3:
* <b> release </b>: R24
* <b> analysis base </b>: '24.2.28'
* <b> campaigns </b>: ['MC23a', 'MC23c']
* <b> architecture </b>: 'RNN'
* <b> datas </b>: ['data']
* <b> backgrounds </b>: ['ttbar', 'wtaunu', 'wmunu', 'wenu', 'ztautau', 'zmumu', 'zee', 'znunu', 'diboson', 'singletop', 'ttX']
* <b> signals </b>: gluinos -> ['GG_XXX_YYY'] & squarks -> ['SS_XXX_YYY']

<br>

<hr>

Run 2:
* <b> release </b>: 'R24'
* <b> analysis base </b>: '24.2.28'
* <b> campaigns </b>: ['MC20a', 'MC20d', 'MC20e']
* <b> architecture </b>: 'RNN'
* <b> datas </b>: ['data']
* <b> backgrounds </b>: ['ttbar', 'wtaunu', 'wmunu', 'wenu', 'ztautau', 'zmumu', 'zee', 'znunu', 'diboson', 'higgs', 'singletop', 'ttX']
* <b> signals </b>: gluinos -> ['GG_XXX_YYY'] & squarks -> ['SS_XXX_YYY']

<br>

<hr>

## Configuration

Loading the Hydra configuration. All analysis parameters (run, region, channel, samples, features) are defined in `configs/` and can be overridden here.

In [4]:
from hydra import compose, initialize_config_dir

initialize_config_dir(config_dir=str(path / "configs"), version_base="1.3")
cfg = compose(config_name="config", overrides=["run@analysis=run2"])

MissingConfigException: In 'config': Could not find 'model/ae'

Config search path:
	provider=hydra, path=pkg://hydra.conf
	provider=main, path=file:///home/islazyk/tau-anomaly-detection/configs
	provider=schema, path=structured://

### Analysis

Constructing the analysis configuration object. 

In [ ]:
from src.processing.analysis import AnalysisConfig

analysis = AnalysisConfig(**cfg.analysis)

### Samples

Building sample lists from config.

In [ ]:
from src.processing.analysis import resolve_samples

samples = resolve_samples(cfg)

## Processing

Reading ROOT ntuples, applying selection cuts (cleaning, channel, RNN, truth-matching, kinematic, region), and computing event weights. Each sample is processed across all campaigns and concatenated.

In [ ]:
from src.processing.processor import process_samples

### Data

Experimental data.

In [ ]:
data = process_samples(
    cfg,
    sample_type="data",
    sample_ids=[s.id for s in samples["data"]],
)

Processing data:   0%|          | 0/3 [00:00<?, ?file/s]

In [ ]:
data

{'data': <Array [{eventClean: 1, ...}, ..., {...}] type='319472 * {eventClean: int8,...'>}

### Background

Monte Carlo-based backgrounds.

In [5]:
background = process_samples(
    cfg,
    sample_type="background",
    sample_ids=[s.id for s in samples["background"]],
)

NameError: name 'process_samples' is not defined

In [6]:
background

NameError: name 'background' is not defined

### Signal

Monte Carlo-based (gluinos and squarks) signals.

In [12]:
signal = process_samples(
    cfg,
    sample_type="signal",
    sample_ids=[s.id for s in samples["signal"]],
)

Processing signal:   0%|          | 0/933 [00:00<?, ?file/s]

File not found, skipping: /disk/atlas3/data_MC/25.2.28/MC20d/vector_XEplateau/GG_1300_100.root
File not found, skipping: /disk/atlas3/data_MC/25.2.28/MC20e/vector_XEplateau/GG_1400_800.root
File not found, skipping: /disk/atlas3/data_MC/25.2.28/MC20e/vector_XEplateau/GG_1500_400.root
File not found, skipping: /disk/atlas3/data_MC/25.2.28/MC20e/vector_XEplateau/GG_2200_600.root
File not found, skipping: /disk/atlas3/data_MC/25.2.28/MC20e/vector_XEplateau/GG_2400_300.root
File not found, skipping: /disk/atlas3/data_MC/25.2.28/MC20e/vector_XEplateau/GG_2600_1300.root


In [13]:
list(signal.keys())[:10]

['GG_1000_700',
 'GG_1000_800_J85_1tau',
 'GG_1000_850_J85_1tau',
 'GG_1000_900_J85_1tau',
 'GG_1000_925_J85_1tau',
 'GG_1000_950_J85_1tau',
 'GG_1000_970_J85_1tau',
 'GG_1050_1020_J85_1tau',
 'GG_1100_100',
 'GG_1100_1000_J85_1tau']

## Merging

Grouping processed samples.

In [14]:
samples = {
    "data": data,
    "background": background,
    "signal": signal,
}

Backgrounds are grouped by the background strategy:
* `as_one`,
* `as_primary`.

In [15]:
from src.processing.merger import merge_backgrounds

samples["background"] = merge_backgrounds(samples["background"], cfg)

In [16]:
samples["background"]

{'topquarks': <Array [{eventClean: 1, ...}, ..., {...}] type='1139269 * {eventClean: int8...'>,
 'wtaunu': <Array [{eventClean: 1, ...}, ..., {...}] type='1716897 * {eventClean: int8...'>,
 'ztautau': <Array [{eventClean: 1, ...}, ..., {...}] type='2314192 * {eventClean: int8...'>,
 'diboson': <Array [{eventClean: 1, ...}, ..., {...}] type='1846720 * {eventClean: int8...'>,
 'other': <Array [{eventClean: 1, ...}, ..., {...}] type='824018 * {eventClean: int8,...'>}

Each event is then tagged with its sample name (`eventOrigin`).

In [17]:
from src.processing.features import assign_event_origin

assign_event_origin(samples)

In [18]:
samples['data']['data']['eventOrigin']

<Array ['data', 'data', 'data', ..., 'data', 'data'] type='319472 * string'>

Signals are grouped by the signal strategy 
* `as_is`,
* `as_one`, 
* `as_type`, 
* `as_mass`.

In [19]:
from src.processing.merger import merge_signals

samples["signal"] = merge_signals(samples["signal"], cfg)

In [20]:
samples

{'data': {'data': <Array [{eventClean: 1, ...}, ..., {...}] type='319472 * {eventClean: int8,...'>},
 'background': {'topquarks': <Array [{eventClean: 1, ...}, ..., {...}] type='1139269 * {eventClean: int8...'>,
  'wtaunu': <Array [{eventClean: 1, ...}, ..., {...}] type='1716897 * {eventClean: int8...'>,
  'ztautau': <Array [{eventClean: 1, ...}, ..., {...}] type='2314192 * {eventClean: int8...'>,
  'diboson': <Array [{eventClean: 1, ...}, ..., {...}] type='1846720 * {eventClean: int8...'>,
  'other': <Array [{eventClean: 1, ...}, ..., {...}] type='824018 * {eventClean: int8,...'>},
 'signal': {'signal': <Array [{eventClean: 1, ...}, ..., {...}] type='1986073 * {eventClean: int8...'>}}

## Serialization

Saving each sample as a parquet file to the path derived from cfg (scope/analysis_base/run/region/channel).

In [21]:
from src.processing.analysis import get_output_paths

samples_dir = path / get_output_paths(cfg)["samples_dir"]

One file per sample key, written separately for data, background, and signal.

In [22]:
from src.processing.io import save_samples

save_samples(samples["data"], samples_dir)
save_samples(samples["background"], samples_dir)
save_samples(samples["signal"], samples_dir)